# Mismatch evolution along parameter axes

Generates detector-frame TDI responses for a 1PAT1R injection and two recovery models
(1PAT1R self-consistent and 0PA_Kerr lmax=5), then plots how the mismatch grows as we
walk away from the injected reference parameters along a single axis (here: primary spin `a`).

All waveform, response and EMRI settings are hard-coded in the first cell — no YAML files
are read at run time.

In [1]:
import sys
# sys.path.insert(0, "../validation/PE_test_runs/src")

import cupy as cp
import numpy as np
import matplotlib.pyplot as plt
from lisaconstants import ASTRONOMICAL_YEAR
from lisaorbits import OEMOrbits
from scipy.signal.windows import tukey

from src.waveform import ResponseConfig, WaveformConfig, build_response, param_names_for
from src.utils import inband_freqs, inner_prod_tdi, mismatch_tdi

/data/leuven/367/vsc36785/miniconda3/envs/emri_env_ddpc/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Hard-coded settings

In [3]:
# ── Detector-frame EMRI injection parameters ──────────────────────────────────
# Masses are already redshifted (detector frame): M_det = M_src * (1 + z), z = 1
Z = 0.3633349514867345
M_SRC  = 733495.4619255429   # [M_sun] source-frame primary
MU_SRC = 73.34954619255429   # [M_sun] source-frame secondary

INJ = dict(
    M      = M_SRC  * (1 + Z),   # detector-frame primary mass [M_sun]
    mu     = MU_SRC * (1 + Z),   # detector-frame secondary mass [M_sun]
    a      = 0.0,             # dimensionless primary spin
    p0     = 15.7905,                 # initial semi-latus rectum [M]
    e0     = 0.0,                 # initial eccentricity
    x_I0   = 1.0,                 # prograde equatorial orbit
    d_L    = 2.0,                 # luminosity distance [Gpc]
    theta_S = 0.5689062462243505, # sky position (source polar) [rad]
    phi_S   = 3.069569972026508,  # sky position (source azimuthal) [rad]
    theta_K = 0.11247032308126413,# spin direction (polar) [rad]
    phi_K   = 0.7524383205999023, # spin direction (azimuthal) [rad]
    Phi_phi0   = 4.809727148143657,
    Phi_theta0 = 0.635376120342196,
    Phi_r0     = 4.2638433178190684,
    chi2   = 0.9,                 # secondary spin (1PAT1R only)
)

# ── Waveform settings ─────────────────────────────────────────────────────────
DT    = 5.0   # sampling cadence [s]
T_WF  = 2.0   # waveform duration [yr]
USE_GPU = True

# ── Response settings ─────────────────────────────────────────────────────────
ORBIT_FILE     = "/data/leuven/367/vsc36785/LISA/Mojito_analysis/esa-trailing-orbits-mojito_validation_test_2.h5"
TDI_GEN        = "2nd generation"
TDI_CHAN       = "XYZ"
ORDER          = 40
OFFSET         = 550.0    # [s] time offset added before/after waveform
N_SAMPLES_DELAY = 1000    # delay samples for TDI
T_BUFFER       = 10000.0  # [s] orbit interpolation buffer
FLIP_HX        = True
IS_ECLIPTIC_LAT = False

# ── Analysis settings ─────────────────────────────────────────────────────────
ALPHA_TUKEY = 0.01   # Tukey window roll-off fraction
F_MIN       = 1e-5   # lower frequency cutoff [Hz]

# ── Mismatch scan settings ────────────────────────────────────────────────────
SCAN_PARAM  = "a"                             # parameter to sweep
DELTA_SCAN  = np.linspace(-0.5, 0.5, 21)      # offsets from injection value

print("Settings loaded.")

Settings loaded.


## Build waveform and response configs

In [4]:
_inspiral_kw  = {"DENSE_STEPPING": 0, "max_init_len": 1000}
_summation_kw = {"pad_output": True}

inj_wcfg = WaveformConfig(
    model="1PAT1R",
    dt=DT,
    T=T_WF,
    evolve_chi1=True,
    include_1PA_amps=True,
    inspiral_kwargs=_inspiral_kw,
    summation_kwargs=_summation_kw,
)

rec_1PA_wcfg = WaveformConfig(
    model="1PAT1R",
    dt=DT,
    T=T_WF,
    evolve_chi1=True,
    include_1PA_amps=True,
    inspiral_kwargs=_inspiral_kw,
    summation_kwargs=_summation_kw,
)

rec_0PA_wcfg = WaveformConfig(
    model="0PA_Kerr",
    dt=DT,
    T=T_WF,
    lmax=5,
    inspiral_kwargs=_inspiral_kw,
    summation_kwargs=_summation_kw,
)

resp_cfg = ResponseConfig(
    orbit_file=ORBIT_FILE,
    tdi_gen=TDI_GEN,
    tdi_chan=TDI_CHAN,
    order=ORDER,
    offset=OFFSET,
    n_samples_delay=N_SAMPLES_DELAY,
    t_buffer=T_BUFFER,
    flip_hx=FLIP_HX,
    is_ecliptic_latitude=IS_ECLIPTIC_LAT,
)

print(f"Injection : {inj_wcfg.model}  evolve_chi1={inj_wcfg.evolve_chi1}  1PA_amps={inj_wcfg.include_1PA_amps}")
print(f"Recovery 1: {rec_1PA_wcfg.model}  evolve_chi1={rec_1PA_wcfg.evolve_chi1}  1PA_amps={rec_1PA_wcfg.include_1PA_amps}")
print(f"Recovery 2: {rec_0PA_wcfg.model}  lmax={rec_0PA_wcfg.lmax}")
print(f"Response  : {resp_cfg.tdi_chan}  {resp_cfg.tdi_gen}  order={resp_cfg.order}")

Injection : 1PAT1R  evolve_chi1=True  1PA_amps=True
Recovery 1: 1PAT1R  evolve_chi1=True  1PA_amps=True
Recovery 2: 0PA_Kerr  lmax=5
Response  : XYZ  2nd generation  order=40


## Timing

In [5]:
oem_orbits = OEMOrbits.from_included("esa-trailing")
t0_orbits  = float(oem_orbits.t_start) + 10.0
t_init     = t0_orbits   # waveform start coincides with orbit epoch

T_response = (
    T_WF
    + (2 * OFFSET + 2 * N_SAMPLES_DELAY * DT) / ASTRONOMICAL_YEAR
)

print(f"t0_orbits  = {t0_orbits:.3f} s")
print(f"t_init     = {t_init:.3f} s")
print(f"T_response = {T_response:.6f} yr")

OEM preferred interpolation method ignored, using spline interpolation (see InterpolatedOrbits for details)


t0_orbits  = 61171239.328 s
t_init     = 61171239.328 s
T_response = 2.000352 yr


## Build detector responses

In [6]:
print("Building injection response (1PAT1R) …")
inj_response = build_response(inj_wcfg, resp_cfg, t_init, t0_orbits, T_response, use_gpu=USE_GPU)

print("Building 1PAT1R recovery response …")
rec_1PA_response = build_response(rec_1PA_wcfg, resp_cfg, t_init, t0_orbits, T_response, use_gpu=USE_GPU)

print("Building 0PA_Kerr recovery response (lmax=5) …")
rec_0PA_response = build_response(rec_0PA_wcfg, resp_cfg, t_init, t0_orbits, T_response, use_gpu=USE_GPU)

print("Done.")

Building injection response (1PAT1R) …
Building 1PAT1R recovery response …
Building 0PA_Kerr recovery response (lmax=5) …
Done.


## Parameter vectors

In [7]:
def emri_vector(inj_dict, model):
    """Build ordered parameter vector for `model` from the INJ dict."""
    return [inj_dict[n] for n in param_names_for(model)]

inj_params      = emri_vector(INJ, "1PAT1R")
rec_1PA_params  = emri_vector(INJ, "1PAT1R")
rec_0PA_params  = emri_vector(INJ, "0PA_Kerr")

# Index of the scan parameter in each model's param vector
pnames_1PA = param_names_for("1PAT1R")
pnames_0PA = param_names_for("0PA_Kerr")
scan_idx_1PA = pnames_1PA.index(SCAN_PARAM)
scan_idx_0PA = pnames_0PA.index(SCAN_PARAM)

print(f"1PAT1R params : {pnames_1PA}")
print(f"0PA_Kerr params: {pnames_0PA}")
print(f"Scan index (1PA / 0PA): {scan_idx_1PA} / {scan_idx_0PA}")

1PAT1R params : ['M', 'mu', 'a', 'p0', 'e0', 'chi2', 'x_I0', 'd_L', 'theta_S', 'phi_S', 'theta_K', 'phi_K', 'Phi_phi0', 'Phi_theta0', 'Phi_r0']
0PA_Kerr params: ['M', 'mu', 'a', 'p0', 'e0', 'x_I0', 'd_L', 'theta_S', 'phi_S', 'theta_K', 'phi_K', 'Phi_phi0', 'Phi_theta0', 'Phi_r0']
Scan index (1PA / 0PA): 2 / 2


## Generate injection + window + FFT

In [8]:
print("Generating injection data …")
xyz_data = inj_response(*inj_params)   # shape: (3, N_t)
N_t = xyz_data.shape[1]
print(f"  N_t = {N_t}")

# ── Tukey window (explicit construction) ──────────────────────────────────────
# The Tukey window is a cosine-tapered rectangle:
#   - flat over the central (1 - alpha) fraction of the signal
#   - smoothly tapers to zero at both ends over alpha/2 of the total length
# Using alpha=0.01 leaves 99 % of samples unweighted while eliminating
# spectral leakage from the sharp on/off at the time-series edges.
window_np = tukey(N_t, alpha=ALPHA_TUKEY)
window    = cp.asarray(window_np)      # move to GPU

# ── Frequency grid + in-band mask ─────────────────────────────────────────────
freqs_inband, mask = inband_freqs(N_t, DT, f_min=F_MIN, filter_freq=True)
n_f = int(mask.sum())
print(f"  n_inband = {n_f}")

# ── FFT injection ─────────────────────────────────────────────────────────────
xyz_data_fft = cp.fft.rfft(xyz_data * window, axis=1)[:, mask]

Generating injection data …
  N_t = 12625479
  n_inband = 6312108


## Analytic LISA TDI PSD and inverse covariance

We use the analytic SciRD LISA noise model to build a diagonal inverse-covariance
matrix, avoiding any dependency on specific Mojito noise files.  The matrix carries
the `2 df = 2 / (N dt)` likelihood normalisation expected by `inner_prod_tdi`.

In [9]:
def lisa_tdi_x_psd(f):
    """Analytic LISA TDI-X one-sided PSD [1/Hz] (simplified SciRD model)."""
    L      = 2.5e9          # arm length [m]
    c      = 3e8            # speed of light [m/s]
    f_star = c / (2 * cp.pi * L)   # transfer frequency ~19.1 mHz
    S_acc  = 9e-30 * (1 + (4e-4 / f)**2) * (1 + (f / 8e-3)**4)
    S_oms  = 2.25e-22 * (1 + (2e-3 / f)**4)
    return (20.0 / 3.0) * (S_oms / L**2 + 2 * S_acc / (2*cp.pi*f)**4 / L**2) * (1 + (f/f_star)**2)

psd_x = lisa_tdi_x_psd(freqs_inband)   # shape: (n_f,)

# Diagonal inv_cov: equal uncorrelated PSDs on X, Y, Z.
# Carries 2 df = 2 / (N_t * DT) likelihood normalisation.
inv_cov = cp.zeros((n_f, 3, 3), dtype=cp.complex128)
df      = 1.0 / (N_t * DT)
for i in range(3):
    inv_cov[:, i, i] = (2.0 * df) / psd_x

print(f"inv_cov shape : {inv_cov.shape}")
print(f"df            = {df:.3e} Hz")

inv_cov shape : (6312108, 3, 3)
df            = 1.584e-08 Hz


## Sanity checks at truth parameters

In [10]:
# Recovery waveforms at the injected reference parameters
xyz_1PA_truth = rec_1PA_response(*rec_1PA_params)
xyz_0PA_truth = rec_0PA_response(*rec_0PA_params)

xyz_1PA_fft_truth = cp.fft.rfft(xyz_1PA_truth * window, axis=1)[:, mask]
xyz_0PA_fft_truth = cp.fft.rfft(xyz_0PA_truth * window, axis=1)[:, mask]

snr         = float(cp.sqrt(inner_prod_tdi(xyz_data_fft, xyz_data_fft, inv_cov)))
mm_1PA_ref  = mismatch_tdi(xyz_data_fft, xyz_1PA_fft_truth, inv_cov)
mm_0PA_ref  = mismatch_tdi(xyz_data_fft, xyz_0PA_fft_truth, inv_cov)

print(f"SNR                                     = {snr:.2f}")
print(f"Mismatch (1PAT1R at truth parameters)   = {mm_1PA_ref:.3e}")
print(f"Mismatch (0PA_Kerr at truth parameters) = {mm_0PA_ref:.3e}")
if snr < 20:
    print("WARNING: SNR below 20.")

/data/leuven/367/vsc36785/miniconda3/envs/emri_env_ddpc/lib/python3.12/site-packages/fastlisaresponse/response.py:645: SyntaxWarning: invalid escape sequence '\p'
  """Wrapper to produce LISA TDI from TD waveforms


TypeError: AmpInterpKerrEccEq.get_amplitudes() got an unexpected keyword argument 'nu'

## Mismatch scan along primary spin axis

We fix all parameters at the injection values and walk away along the `a` axis,
computing the mismatch between the fixed injection and each recovery template.

In [ ]:
a_ref   = INJ[SCAN_PARAM]
a_scan  = a_ref + DELTA_SCAN

mm_1PA_scan = []
mm_0PA_scan = []

for k, a_val in enumerate(a_scan):
    # 1PAT1R recovery template
    p1PA = list(rec_1PA_params)
    p1PA[scan_idx_1PA] = a_val
    xyz_1PA = rec_1PA_response(*p1PA)
    fft_1PA = cp.fft.rfft(xyz_1PA * window, axis=1)[:, mask]
    mm_1PA_scan.append(mismatch_tdi(xyz_data_fft, fft_1PA, inv_cov))

    # 0PA_Kerr recovery template
    p0PA = list(rec_0PA_params)
    p0PA[scan_idx_0PA] = a_val
    xyz_0PA = rec_0PA_response(*p0PA)
    fft_0PA = cp.fft.rfft(xyz_0PA * window, axis=1)[:, mask]
    mm_0PA_scan.append(mismatch_tdi(xyz_data_fft, fft_0PA, inv_cov))

    if (k + 1) % 5 == 0:
        print(f"  {k+1}/{len(a_scan)}  a={a_val:.4f}  "
              f"mm_1PA={mm_1PA_scan[-1]:.3e}  mm_0PA={mm_0PA_scan[-1]:.3e}")

mm_1PA_scan = np.array(mm_1PA_scan)
mm_0PA_scan = np.array(mm_0PA_scan)

print("Scan complete.")

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.semilogy(DELTA_SCAN, mm_1PA_scan, "o-", color="C0", label="1PAT1R recovery")
ax.semilogy(DELTA_SCAN, mm_0PA_scan, "s-", color="C1", label="0PA Kerr recovery (lmax=5)")
ax.axvline(0, color="k", ls="--", lw=0.8, label=f"Injection ($a={a_ref:.5f}$)")
ax.axhline(1.0 / (2.0 * snr**2), color="gray", ls=":", lw=1.0,
           label=r"$1/(2\rho^2)$ faithfulness threshold")

ax.set_xlabel(r"$\Delta a$ (offset from injected primary spin)")
ax.set_ylabel("Mismatch $\\mathcal{M}$")
ax.set_title(
    f"Mismatch vs primary spin offset\n"
    f"Injection: 1PAT1R, SNR = {snr:.1f}, "
    f"$M={INJ['M']:.2e}\\,M_\\odot$, $\\mu={INJ['mu']:.1f}\\,M_\\odot$"
)
ax.legend()
ax.grid(True, which="both", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# FD diagnostic: injection vs both recovery templates at truth parameters
freqs_np = cp.asnumpy(freqs_inband)
data_np  = cp.asnumpy(xyz_data_fft)
rec1_np  = cp.asnumpy(xyz_1PA_fft_truth)
rec0_np  = cp.asnumpy(xyz_0PA_fft_truth)

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(freqs_np, 2 * freqs_np * np.abs(data_np[0]), label="Injection (1PAT1R)", alpha=0.85)
ax.loglog(freqs_np, 2 * freqs_np * np.abs(rec1_np[0]), label="1PAT1R at truth",   alpha=0.8, ls="--")
ax.loglog(freqs_np, 2 * freqs_np * np.abs(rec0_np[0]), label="0PA Kerr at truth", alpha=0.8, ls=":")
ax.loglog(freqs_np, np.sqrt(freqs_np * cp.asnumpy(psd_x)), color="k", lw=0.8, label="Noise ASD (X)")
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Characteristic strain $2f|\\tilde{h}(f)|$")
ax.set_title("FD comparison — X channel at truth parameters")
ax.legend()
ax.grid(True, which="both", alpha=0.4)
plt.tight_layout()
plt.show()

## Schwarzschild model comparison at a = 0

We compare the 1PAT1R waveform evaluated at exact Schwarzschild spin (`a = 0`) against
two legacy Schwarzschild 1PA FEW models:

* `FastSchwarzschildEccentricFluxBicubic` — bicubic-interpolated flux tables
* `FastSchwarzschildEccentricFlux`        — direct flux evaluation

Because these models have no spin degree of freedom, the parameter vector drops `a`
entirely.  The four mismatches computed are:

| Pair | Description |
|------|-------------|
| 1PAT1R – Bicubic   | 1PA Kerr (a=0) vs 1PA Schwarzschild (bicubic) |
| 1PAT1R – Flux      | 1PA Kerr (a=0) vs 1PA Schwarzschild (direct)  |
| 1PAT1R – 0PA_Kerr  | 1PA vs 0PA at a=0 |
| Bicubic – Flux     | Cross-check between the two Schwarzschild backends |

In [ ]:
from few.waveform import GenerateEMRIWaveform
from fastlisaresponse import ResponseWrapper
from fastlisaresponse.tdiconfig import TDIConfig
from fastlisaresponse.utils.parallelbase import ParallelModuleBase
from lisatools.detector import Orbits

# Schwarzschild parameter names: same as 0PA_Kerr but without 'a'
PARAM_NAMES_SCHW = [
    'M', 'mu', 'p0', 'e0', 'x_I0', 'd_L',
    'theta_S', 'phi_S', 'theta_K', 'phi_K',
    'Phi_phi0', 'Phi_theta0', 'Phi_r0',
]


class SchwWave(ParallelModuleBase):
    """Thin wrapper around a Schwarzschild FEW waveform (no spin parameter)."""

    def __init__(self, model_name, T_waveform, dt, inspiral_kwargs=None):
        self.min_output_length = 0
        self._gen = GenerateEMRIWaveform(
            model_name,
            return_list=False,
            inspiral_kwargs=inspiral_kwargs or {},
            frame="detector",
        )
        self.T   = T_waveform
        self.dt  = dt

    @classmethod
    def supported_backends(cls):
        return ["fastlisaresponse_" + b for b in cls.GPU_RECOMMENDED()]

    def __call__(self, *params, **kw):
        h = self._gen(*params, dt=self.dt, T=self.T, pad_output=True)
        if self.min_output_length > 0 and len(h) < self.min_output_length:
            xp = cp.get_array_module(h)
            h  = xp.concatenate([h, xp.zeros(self.min_output_length - len(h), dtype=h.dtype)])
        return h


def build_schw_response(model_name, resp_cfg, t_init, t0_orbits, T_response, use_gpu=True):
    """Build a TDI response callable for a Schwarzschild FEW model."""
    force_backend = "cuda12x" if use_gpu else None

    wave   = SchwWave(model_name, T_response, DT, _inspiral_kw)
    orbits = Orbits(
        filename=resp_cfg.orbit_file,
        use_gpu=use_gpu,
        force_backend=force_backend,
        linear_interp_setup=False,
        t0=t0_orbits,
    )
    tdi_kwargs = dict(
        orbits=orbits,
        order=resp_cfg.order,
        tdi=TDIConfig(resp_cfg.tdi_gen),
        tdi_chan=resp_cfg.tdi_chan,
    )
    response = ResponseWrapper(
        wave,
        T_response,
        DT,
        7,   # index_lambda = phi_S
        6,   # index_beta   = theta_S
        t0=t_init,
        t_buffer=resp_cfg.t_buffer,
        flip_hx=resp_cfg.flip_hx,
        force_backend=force_backend,
        remove_sky_coords=resp_cfg.remove_sky_coords,
        is_ecliptic_latitude=resp_cfg.is_ecliptic_latitude,
        remove_garbage=resp_cfg.remove_garbage,
        **tdi_kwargs,
    )
    wave.min_output_length = response.response_model.num_pts

    def _call(*params):
        return cp.asarray(response(*params))

    return _call


print("Building FastSchwarzschildEccentricFluxBicubic response …")
rec_bicubic_response = build_schw_response(
    "FastSchwarzschildEccentricFluxBicubic",
    resp_cfg, t_init, t0_orbits, T_response, use_gpu=USE_GPU,
)

print("Building FastSchwarzschildEccentricFlux response …")
rec_flux_response = build_schw_response(
    "FastSchwarzschildEccentricFlux",
    resp_cfg, t_init, t0_orbits, T_response, use_gpu=USE_GPU,
)

# ── Parameter vectors at a = 0 ────────────────────────────────────────────────
# 1PAT1R: set a = 0 (index 2 in PARAM_NAMES_1PA)
params_1PA_a0 = list(inj_params)
params_1PA_a0[param_names_for("1PAT1R").index("a")] = 0.0

# 0PA_Kerr: set a = 0 (index 2 in PARAM_NAMES_0PA)
params_0PA_a0 = list(rec_0PA_params)
params_0PA_a0[param_names_for("0PA_Kerr").index("a")] = 0.0

# Schwarzschild: no spin, build vector from injection dict
schw_params = [INJ[n] for n in PARAM_NAMES_SCHW]

print("Done building responses and parameter vectors.")

In [ ]:
def nonzero_length(xyz_gpu):
    """Number of samples before the trailing zero-pad in a (3, N_t) TDI array."""
    x = cp.asnumpy(xyz_gpu[0])
    nz = np.flatnonzero(x)
    return int(nz[-1]) + 1 if len(nz) > 0 else x.shape[0]


# ── Generate all waveforms ────────────────────────────────────────────────────
print("Generating 1PAT1R at a=0 …")
xyz_1PA_a0  = rec_1PA_response(*params_1PA_a0)

print("Generating 0PA_Kerr at a=0 …")
xyz_0PA_a0  = rec_0PA_response(*params_0PA_a0)

print("Generating FastSchwarzschildEccentricFluxBicubic …")
xyz_bicubic = rec_bicubic_response(*schw_params)

print("Generating FastSchwarzschildEccentricFlux …")
xyz_flux    = rec_flux_response(*schw_params)

# ── Fair window: Tukey over the shortest non-zero region, zero-padded ─────────
# This ensures we only compare samples where ALL models have signal, making
# the mismatch insensitive to differences in inspiral duration.
lens = {
    "1PAT1R (a=0)"          : nonzero_length(xyz_1PA_a0),
    "0PA_Kerr (a=0)"        : nonzero_length(xyz_0PA_a0),
    "Bicubic"               : nonzero_length(xyz_bicubic),
    "Flux"                  : nonzero_length(xyz_flux),
}
for name, l in lens.items():
    print(f"  non-zero length  {name:30s}: {l} samples  ({l * DT / ASTRONOMICAL_YEAR:.4f} yr)")

min_len = min(lens.values())
print(f"\nFair window length: {min_len} samples  ({min_len * DT / ASTRONOMICAL_YEAR:.4f} yr)")

window_fair_np        = np.zeros(N_t)
window_fair_np[:min_len] = tukey(min_len, alpha=ALPHA_TUKEY)
window_fair           = cp.asarray(window_fair_np)

# ── FFT with fair window ──────────────────────────────────────────────────────
xyz_1PA_a0_fft  = cp.fft.rfft(xyz_1PA_a0  * window_fair, axis=1)[:, mask]
xyz_0PA_a0_fft  = cp.fft.rfft(xyz_0PA_a0  * window_fair, axis=1)[:, mask]
xyz_bicubic_fft = cp.fft.rfft(xyz_bicubic * window_fair, axis=1)[:, mask]
xyz_flux_fft    = cp.fft.rfft(xyz_flux    * window_fair, axis=1)[:, mask]

# ── Mismatches ────────────────────────────────────────────────────────────────
mm_1PA_bicubic  = mismatch_tdi(xyz_1PA_a0_fft, xyz_bicubic_fft, inv_cov)
mm_1PA_flux     = mismatch_tdi(xyz_1PA_a0_fft, xyz_flux_fft,    inv_cov)
mm_1PA_0PA_a0   = mismatch_tdi(xyz_1PA_a0_fft, xyz_0PA_a0_fft,  inv_cov)
mm_bicubic_flux = mismatch_tdi(xyz_bicubic_fft, xyz_flux_fft,   inv_cov)

print("\nMismatch summary (fair window, a = 0, truth parameters):")
print(f"  1PAT1R  vs  FastSchwarzschildEccentricFluxBicubic : {mm_1PA_bicubic:.4e}")
print(f"  1PAT1R  vs  FastSchwarzschildEccentricFlux        : {mm_1PA_flux:.4e}")
print(f"  1PAT1R  vs  0PA_Kerr (a=0)                        : {mm_1PA_0PA_a0:.4e}")
print(f"  FastSchwarzschildEccentricFluxBicubic vs Flux      : {mm_bicubic_flux:.4e}")

In [ ]:
def fd_comparison_plot(fft_a, fft_b, label_a, label_b, freqs, psd, mismatch=None):
    """
    Two-panel frequency-domain comparison.

    Top panel   : characteristic strain of both models + LISA noise curve.
    Bottom panel: characteristic strain of the residual + LISA noise curve.
    Mismatch value is shown in the title if provided.
    """
    f   = cp.asnumpy(freqs)
    a_X = cp.asnumpy(fft_a[0])
    b_X = cp.asnumpy(fft_b[0])
    res_X = a_X - b_X
    noise = np.sqrt(f * cp.asnumpy(psd))

    mm_str = f"  |  mismatch = {mismatch:.3e}" if mismatch is not None else ""
    title  = f"{label_a}  vs  {label_b}{mm_str}"

    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True,
                             gridspec_kw={"hspace": 0.08})

    ax0 = axes[0]
    ax0.loglog(f, 2 * f * np.abs(a_X), label=label_a,  alpha=0.9, lw=1.2)
    ax0.loglog(f, 2 * f * np.abs(b_X), label=label_b,  alpha=0.8, lw=1.2, ls="--")
    ax0.loglog(f, noise, color="k", lw=0.9, ls=":",     label="LISA noise $\\sqrt{f S_n(f)}$ (X)")
    ax0.set_ylabel("Characteristic strain $2f|\\tilde{h}(f)|$")
    ax0.set_title(title, fontsize=10)
    ax0.legend(fontsize=8)
    ax0.grid(True, which="both", alpha=0.35)

    ax1 = axes[1]
    ax1.loglog(f, 2 * f * np.abs(res_X), color="C2", lw=1.2, label="Residual $|h_A - h_B|$")
    ax1.loglog(f, noise, color="k", lw=0.9, ls=":",           label="LISA noise $\\sqrt{f S_n(f)}$ (X)")
    ax1.set_xlabel("Frequency [Hz]")
    ax1.set_ylabel("Characteristic strain $2f|\\tilde{h}(f)|$")
    ax1.legend(fontsize=8)
    ax1.grid(True, which="both", alpha=0.35)

    plt.tight_layout()
    plt.show()
    return fig


# ── One plot per pair ─────────────────────────────────────────────────────────
fd_comparison_plot(
    xyz_1PA_a0_fft, xyz_bicubic_fft,
    "1PAT1R (a=0)", "FastSchwarzschildEccentricFluxBicubic",
    freqs_inband, psd_x, mismatch=mm_1PA_bicubic,
)

fd_comparison_plot(
    xyz_1PA_a0_fft, xyz_flux_fft,
    "1PAT1R (a=0)", "FastSchwarzschildEccentricFlux",
    freqs_inband, psd_x, mismatch=mm_1PA_flux,
)

fd_comparison_plot(
    xyz_1PA_a0_fft, xyz_0PA_a0_fft,
    "1PAT1R (a=0)", "0PA_Kerr (a=0)",
    freqs_inband, psd_x, mismatch=mm_1PA_0PA_a0,
)

fd_comparison_plot(
    xyz_bicubic_fft, xyz_flux_fft,
    "FastSchwarzschildEccentricFluxBicubic", "FastSchwarzschildEccentricFlux",
    freqs_inband, psd_x, mismatch=mm_bicubic_flux,
)